In [12]:
import torch

sizes = [3, 4, 5]
N_total = sum(sizes)  # 12
A = torch.zeros(N_total, N_total)

# 把每张图的邻接矩阵放到对角线上
cur = 0
for n in sizes:
    # 简单 5 节点图: 全连接
    block = torch.randint(0, 2, (n, n))
    print(block)
    print(block.t())
    print(block+block.t()>0)
    block = ((block + block.t()) > 0).float()
    print(block)  # 对称
    block.fill_diagonal_(0)
    print(block)  # 去掉自环
    A[cur:cur+n, cur:cur+n] = block
    cur += n


tensor([[1, 0, 0],
        [1, 1, 0],
        [0, 0, 1]])
tensor([[1, 1, 0],
        [0, 1, 0],
        [0, 0, 1]])
tensor([[ True,  True, False],
        [ True,  True, False],
        [False, False,  True]])
tensor([[1., 1., 0.],
        [1., 1., 0.],
        [0., 0., 1.]])
tensor([[0., 1., 0.],
        [1., 0., 0.],
        [0., 0., 0.]])
tensor([[1, 1, 1, 0],
        [1, 1, 0, 0],
        [1, 0, 0, 0],
        [0, 1, 0, 0]])
tensor([[1, 1, 1, 0],
        [1, 1, 0, 1],
        [1, 0, 0, 0],
        [0, 0, 0, 0]])
tensor([[ True,  True,  True, False],
        [ True,  True, False,  True],
        [ True, False, False, False],
        [False,  True, False, False]])
tensor([[1., 1., 1., 0.],
        [1., 1., 0., 1.],
        [1., 0., 0., 0.],
        [0., 1., 0., 0.]])
tensor([[0., 1., 1., 0.],
        [1., 0., 0., 1.],
        [1., 0., 0., 0.],
        [0., 1., 0., 0.]])
tensor([[1, 0, 1, 0, 1],
        [1, 1, 1, 0, 1],
        [0, 1, 1, 0, 1],
        [1, 1, 1, 0, 1],
        [0, 0, 

In [14]:
import torch

A = torch.tensor([[0, 1, 0, 0],
                  [1, 0, 1, 0],
                  [0, 1, 0, 1],
                  [0, 0, 1, 0]], dtype=torch.float)

A_hat = A + torch.eye(A.shape[0])
print('A_hat =')
print(A_hat.int())

A_hat =
tensor([[1, 1, 0, 0],
        [1, 1, 1, 0],
        [0, 1, 1, 1],
        [0, 0, 1, 1]], dtype=torch.int32)


In [16]:
import torch

A_hat = torch.tensor([[1, 1, 0, 0],
                       [1, 2, 1, 0],
                       [0, 1, 2, 1],
                       [0, 0, 1, 1]], dtype=torch.float)

# 每行求和 = 度
deg = A_hat.sum(dim=1)
print(deg)

# clamp 避免除零
deg = deg.clamp(min=1e-12)

print('Degree vector:', deg)
print('Total edges × 2 (含自环):', A_hat.sum().item() / 2)

tensor([2., 4., 4., 2.])
Degree vector: tensor([2., 4., 4., 2.])
Total edges × 2 (含自环): 6.0


In [17]:
import torch

deg = torch.tensor([1., 2., 2., 1.])

d_inv_sqrt = deg.pow(-0.5)
print('d_inv_sqrt:', d_inv_sqrt)
print('期望: [1.0, 0.707, 0.707, 1.0]')

d_inv_sqrt: tensor([1.0000, 0.7071, 0.7071, 1.0000])
期望: [1.0, 0.707, 0.707, 1.0]


In [18]:
import torch

A_hat = torch.tensor([[1, 1, 0, 0],
                       [1, 2, 1, 0],
                       [0, 1, 2, 1],
                       [0, 0, 1, 1]], dtype=torch.float)
deg = A_hat.sum(dim=-1).clamp(min=1e-12)
d_inv_sqrt = deg.pow(-0.5)
D = torch.diag_embed(d_inv_sqrt)

# 对称归一化: D^{-1/2} A_hat D^{-1/2}
A_norm = D @ A_hat @ D

print('A_norm =')
print(A_norm)
print(f'A_norm 每行和: {A_norm.sum(dim=-1).tolist()}')

A_norm =
tensor([[0.5000, 0.3536, 0.0000, 0.0000],
        [0.3536, 0.5000, 0.2500, 0.0000],
        [0.0000, 0.2500, 0.5000, 0.3536],
        [0.0000, 0.0000, 0.3536, 0.5000]])
A_norm 每行和: [0.8535533547401428, 1.1035534143447876, 1.1035534143447876, 0.8535533547401428]


In [19]:
import torch

B, N, F_in, F_out = 2, 4, 3, 5
x = torch.randn(B, N, F_in)
W = torch.randn(F_in, F_out)

support = x @ W
print('support shape:', support.shape)
print('期望: torch.Size([2, 4, 5])')

support shape: torch.Size([2, 4, 5])
期望: torch.Size([2, 4, 5])


In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GCNLayer(nn.Module):
    def __init__(self, F_in, F_out):
        super().__init__()
        self.W = nn.Parameter(torch.randn(F_in, F_out) * 0.1)
        self.b = nn.Parameter(torch.zeros(F_out))

    def forward(self, x, a_norm):
        support = x @ self.W    # 1) 线性变换
        out = a_norm @ support        # 2) 邻居聚合
        out = out + self.b        # 3) 加 bias
        return F.relu(out)      # 4) 激活

# 测试
layer = GCNLayer(3, 5)
x = torch.randn(2, 4, 3)
A = torch.eye(4).unsqueeze(0).expand(2, -1, -1)
out = layer(x, A)
print('Output shape:', out.shape)
print('Output > 0 (after ReLU):', (out > 0).all().item())

Output shape: torch.Size([2, 4, 5])
Output > 0 (after ReLU): False


In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 复用刚才的 GCNLayer
class GCNLayer(nn.Module):
    def __init__(self, F_in, F_out):
        super().__init__()
        self.W = nn.Parameter(torch.randn(F_in, F_out) * 0.1)
        self.b = nn.Parameter(torch.zeros(F_out))
    def forward(self, x, a_norm):
        return F.relu(a_norm @ (x @ self.W) + self.b)

class GCNClassifier(nn.Module):
    def __init__(self, F_in, hidden, n_classes):
        super().__init__()
        self.conv1 = GCNLayer(F_in, hidden)
        self.conv2 = GCNLayer(hidden, hidden)
        self.classifier = nn.Linear(hidden, n_classes)

    def forward(self, x, a_norm):
        h = F.relu(self.conv1(x, a_norm))
        h = F.relu(self.conv2(h, a_norm))
        # mean pool over nodes
        g = h.mean(dim=1)    # 沿哪个维度平均?
        return self.classifier(g)

# 测试
model = GCNClassifier(F_in=3, hidden=8, n_classes=2)
x = torch.randn(2, 5, 3)
A = torch.eye(5).unsqueeze(0).expand(2, -1, -1)
logits = model(x, A)
print('Logits shape:', logits.shape)
print('Logits:', logits)

Logits shape: torch.Size([2, 2])
Logits: tensor([[ 0.3514, -0.3574],
        [ 0.3470, -0.3626]], grad_fn=<AddmmBackward0>)


In [26]:
import torch

B, N, F = 2, 4, 3
x = torch.randn(B, N, F)
# 简单邻接矩阵 (含自环)
adj = torch.eye(N).unsqueeze(0).expand(B, -1, -1) + \
      torch.diag(torch.ones(N-1), 1).unsqueeze(0).expand(B, -1, -1) + \
      torch.diag(torch.ones(N-1), -1).unsqueeze(0).expand(B, -1, -1)

# 算度
deg = adj.sum(dim=-1, keepdim=True).clamp(min=1.0)

# 邻居平均: 不用对称归一化
neighbor_mean = adj @ x / deg
print('Neighbor mean shape:', neighbor_mean.shape)
print(neighbor_mean[0, 0])  # 第 1 张图第 0 个节点的邻居平均

Neighbor mean shape: torch.Size([2, 4, 3])
tensor([ 0.2322,  1.3265, -1.5482])


In [27]:
import torch

B, N, F = 2, 4, 3
x = torch.randn(B, N, F)
neighbor_mean = torch.randn(B, N, F)  # 假装是上一步的结果

# 拼接自身 + 邻居平均
h = torch.cat([x, neighbor_mean], dim=-1)
print('After CONCAT shape:', h.shape)
print(f'特征维度应该是 2*F = {2*F}')

After CONCAT shape: torch.Size([2, 4, 6])
特征维度应该是 2*F = 6


In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SAGEConv(nn.Module):
    def __init__(self, F_in, F_out):
        super().__init__()
        # 注意：因为 CONCAT 翻倍了，W 输入是 2*F_in
        self.W = nn.Parameter(torch.randn(2 * F_in, F_out) * 0.1)
        self.b = nn.Parameter(torch.zeros(F_out))

    def forward(self, x, adj):
        deg = adj.sum(-1, keepdim=True).clamp(min=1.0)
        neighbor_mean = adj @ x / deg
        h = torch.cat([x, neighbor_mean], dim=-1)  # CONCAT
        return h @ self.W + self.b   # 线性变换 + bias

# 测试
layer = SAGEConv(F_in=3, F_out=5)
x = torch.randn(2, 4, 3)
A = torch.eye(4).unsqueeze(0).expand(2, -1, -1)
out = layer(x, A)
print('Output shape:', out.shape)
print('Output sample:', out[0, 0])

Output shape: torch.Size([2, 4, 5])
Output sample: tensor([ 0.3018, -0.1690, -0.1888, -0.6010,  0.1400],
       grad_fn=<SelectBackward0>)


In [30]:
import torch
import torch.nn.functional as F

B, N, F_in, F_out = 2, 4, 3, 4
H = 2  # 2 heads
x = torch.randn(B, N, F_in)
W = torch.randn(F_in, H * F_out) * 0.1
a_l = torch.randn(H, F_out, 1) * 0.1
a_r = torch.randn(H, F_out, 1) * 0.1

# 线性变换
h = (x @ W).view(B, N, H, F_out)

# 算 f_l, f_r: [B, N, H]
f_l = (h * a_l.view(1, 1, H, F_out)).sum(dim=-1)
f_r = (h * a_r.view(1, 1, H, F_out)).sum(dim=-1)

# 拼成 e_uv: [B, H, N, N]
# 关键: e[u][v] = f_l[v] + f_r[u]
e = f_l.unsqueeze(2) + f_r.unsqueeze(1)
e = e.permute(0, 3, 1, 2)

print('e shape:', e.shape)
print('e[0, 0]:', e[0, 0])

e shape: torch.Size([2, 2, 4, 4])
e[0, 0]: tensor([[-0.2690, -0.1320, -0.0633,  0.0221],
        [-0.2782, -0.1411, -0.0725,  0.0129],
        [-0.2399, -0.1029, -0.0342,  0.0512],
        [-0.2364, -0.0993, -0.0307,  0.0547]])
